# Code Smell Detection Pipeline

Run 14 instruction-tuned coding models on the code smell dataset, then detect smells in the generated code.

**Setup:**
1. Runtime → Change runtime type → **T4 GPU** (free) or **A100** (Pro)
2. Run cells top to bottom
3. Results auto-save to Google Drive

**VRAM Guide:**
- T4 (16GB): runs ≤3B natively, 7B with 4-bit quantization
- A100 (40GB): runs all models natively
- A100 (80GB): runs everything with headroom

## 1. Setup & Install

In [ ]:
# Mount Google Drive for persistent storage
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Install dependencies
!pip install -q torch transformers accelerate bitsandbytes sentencepiece protobuf huggingface_hub

In [ ]:
# Clone the repo
!git clone https://github.com/rosawoo/code-smell.git /content/code-smell 2>/dev/null || echo 'Repo already cloned'
%cd /content/code-smell

In [ ]:
# Check GPU
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
else:
    raise RuntimeError("No GPU detected! Go to Runtime > Change runtime type > T4 GPU")

In [ ]:
# Create persistent output directory on Google Drive
import os
import shutil

DRIVE_OUTPUT = '/content/drive/MyDrive/code-smell-outputs'
LOCAL_OUTPUT = '/content/code-smell/outputs'

os.makedirs(DRIVE_OUTPUT, exist_ok=True)
os.makedirs(LOCAL_OUTPUT, exist_ok=True)

# Restore any previous results from Drive
if os.path.exists(DRIVE_OUTPUT):
    for item in os.listdir(DRIVE_OUTPUT):
        src = os.path.join(DRIVE_OUTPUT, item)
        dst = os.path.join(LOCAL_OUTPUT, item)
        if os.path.isdir(src) and not os.path.exists(dst):
            shutil.copytree(src, dst)
            print(f"Restored previous results: {item}")

print(f"Drive output: {DRIVE_OUTPUT}")
print(f"Local output: {LOCAL_OUTPUT}")

## 2. Model Configuration

Detect VRAM and determine which models can run natively vs. quantized.

In [ ]:
import sys
sys.path.insert(0, '/content/code-smell')

from pipeline.model_registry import MODELS, INFERENCE_CONFIG, SYSTEM_PROMPT

# Detect VRAM and plan quantization
vram_gb = torch.cuda.get_device_properties(0).total_mem / 1e9

def get_quantize_plan(model_config, vram_gb):
    """Determine if a model needs quantization for available VRAM."""
    params = model_config['params']
    # Parse parameter count
    num = float(params.replace('B', ''))
    if model_config['dense_or_moe'] == 'moe':
        num = float(model_config['params_active'].replace('B', ''))

    bf16_mem = num * 2  # ~2 bytes per param in BF16
    q4_mem = num * 0.6  # ~0.6 bytes per param in 4-bit

    if bf16_mem + 2 < vram_gb:  # +2GB headroom
        return None  # BF16 fits
    elif q4_mem + 2 < vram_gb:
        return '4bit'
    else:
        return 'skip'  # Won't fit even quantized

print(f"Available VRAM: {vram_gb:.1f} GB\n")
print(f"{'Model':<35} {'Params':>8} {'Plan':>10}")
print('-' * 58)

run_plan = []
for m in MODELS:
    plan = get_quantize_plan(m, vram_gb)
    status = plan if plan else 'BF16'
    if plan == 'skip':
        status = 'SKIP (too large)'
    print(f"{m['id']:<35} {m['params']:>8} {status:>10}")
    if plan != 'skip':
        run_plan.append((m, plan))

print(f"\nWill run: {len(run_plan)} / {len(MODELS)} models")

## 3. Inference Engine

Core functions to load models, generate code, and save results.

In [ ]:
import json
import time
import re
import gc
from datetime import datetime
from pathlib import Path
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

DATASET_PATH = '/content/code-smell/dataset/prompts_core.json'

with open(DATASET_PATH) as f:
    ALL_PROMPTS = json.load(f)
print(f"Loaded {len(ALL_PROMPTS)} prompts")


def load_completed_ids(model_id):
    """Check which prompts are already done (for resume)."""
    results_file = Path(LOCAL_OUTPUT) / model_id / 'results.jsonl'
    completed = set()
    if results_file.exists():
        with open(results_file) as f:
            for line in f:
                try:
                    completed.add(json.loads(line)['prompt_id'])
                except (json.JSONDecodeError, KeyError):
                    continue
    return completed


def load_model(hf_model_id, quantize=None):
    """Load model + tokenizer."""
    print(f"  Loading {hf_model_id}...")
    tokenizer = AutoTokenizer.from_pretrained(
        hf_model_id, trust_remote_code=True, padding_side='left'
    )
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    load_kwargs = {'trust_remote_code': True, 'device_map': 'auto'}

    if quantize == '4bit':
        load_kwargs['quantization_config'] = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.bfloat16,
            bnb_4bit_use_double_quant=True,
            bnb_4bit_quant_type='nf4',
        )
    else:
        load_kwargs['torch_dtype'] = torch.bfloat16

    model = AutoModelForCausalLM.from_pretrained(hf_model_id, **load_kwargs)
    model.eval()
    print(f"  Loaded on {model.device}")
    return model, tokenizer


def generate_code(model, tokenizer, prompt_text):
    """Generate a single response."""
    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user', 'content': prompt_text},
    ]

    if hasattr(tokenizer, 'apply_chat_template'):
        input_text = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
    else:
        input_text = f"### System:\n{SYSTEM_PROMPT}\n\n### User:\n{prompt_text}\n\n### Assistant:\n"

    inputs = tokenizer(input_text, return_tensors='pt').to(model.device)
    input_len = inputs['input_ids'].shape[1]

    start = time.time()
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=INFERENCE_CONFIG['max_new_tokens'],
            temperature=INFERENCE_CONFIG['temperature'],
            top_p=INFERENCE_CONFIG['top_p'],
            repetition_penalty=INFERENCE_CONFIG['repetition_penalty'],
            do_sample=INFERENCE_CONFIG['do_sample'],
            pad_token_id=tokenizer.pad_token_id,
        )
    elapsed = time.time() - start

    new_tokens = outputs[0][input_len:]
    response = tokenizer.decode(new_tokens, skip_special_tokens=True)

    return {
        'response': response,
        'input_tokens': input_len,
        'output_tokens': len(new_tokens),
        'generation_time_s': round(elapsed, 2),
        'tokens_per_second': round(len(new_tokens) / elapsed, 2) if elapsed > 0 else 0,
    }


def extract_python_code(response):
    """Extract code from markdown fences if present."""
    matches = re.findall(r'```(?:python)?\s*\n(.*?)```', response, re.DOTALL)
    return '\n\n'.join(matches) if matches else response


def sync_to_drive(model_id):
    """Copy model results to Google Drive."""
    src = Path(LOCAL_OUTPUT) / model_id
    dst = Path(DRIVE_OUTPUT) / model_id
    if src.exists():
        if dst.exists():
            shutil.rmtree(dst)
        shutil.copytree(src, dst)


def free_gpu():
    """Free GPU memory between models."""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


print('Inference engine ready.')

## 4. Run All Models

This is the main loop. It:
- Loads each model one at a time
- Generates code for all 426 prompts
- Saves results incrementally (safe to interrupt)
- Syncs to Google Drive after each model
- Frees GPU memory before loading the next model

**Expected time per model (T4 GPU):**
- 1.3-1.5B: ~20-40 min
- 3B: ~40-80 min
- 7B (4-bit): ~80-150 min

If your session disconnects, just re-run — it will resume from where it left off.

In [ ]:
SKIP_MODELS = {
    # Uncomment model IDs to skip them:
    # 'mamba-codestral-7b',  # Requires mamba-ssm package
}

for model_config, quantize in run_plan:
    model_id = model_config['id']
    hf_id = model_config['hf_model_id']

    if model_id in SKIP_MODELS:
        print(f"\n{'='*60}")
        print(f"SKIPPING: {model_id}")
        print(f"{'='*60}")
        continue

    # Check what's already done
    completed_ids = load_completed_ids(model_id)
    remaining = [p for p in ALL_PROMPTS if p['id'] not in completed_ids]

    print(f"\n{'='*60}")
    print(f"MODEL: {model_id} ({model_config['params']})")
    print(f"HuggingFace: {hf_id}")
    print(f"Quantization: {quantize or 'BF16'}")
    print(f"Progress: {len(completed_ids)}/{len(ALL_PROMPTS)} done, {len(remaining)} remaining")
    print(f"{'='*60}")

    if not remaining:
        print("Already complete! Skipping.")
        continue

    # Setup output dirs
    model_dir = Path(LOCAL_OUTPUT) / model_id
    model_dir.mkdir(parents=True, exist_ok=True)
    (model_dir / 'code').mkdir(exist_ok=True)

    # Load model
    try:
        model, tokenizer = load_model(hf_id, quantize)
    except Exception as e:
        print(f"  FAILED to load: {e}")
        free_gpu()
        continue

    # Save metadata
    metadata = {
        'model_id': model_id,
        'hf_model_id': hf_id,
        'family': model_config['family'],
        'architecture': model_config['architecture'],
        'params': model_config['params'],
        'params_active': model_config['params_active'],
        'dense_or_moe': model_config['dense_or_moe'],
        'inference_config': INFERENCE_CONFIG,
        'quantization': quantize,
        'system_prompt': SYSTEM_PROMPT,
        'gpu': torch.cuda.get_device_name(0),
        'vram_gb': round(vram_gb, 1),
        'run_started': datetime.now().isoformat(),
        'total_prompts': len(ALL_PROMPTS),
    }
    with open(model_dir / 'metadata.json', 'w') as f:
        json.dump(metadata, f, indent=2)

    # Run inference
    results_file = model_dir / 'results.jsonl'
    errors = []

    for i, prompt in enumerate(remaining):
        prompt_id = prompt['id']
        print(f"  [{len(completed_ids)+i+1}/{len(ALL_PROMPTS)}] {prompt_id} ", end='', flush=True)

        try:
            result = generate_code(model, tokenizer, prompt['prompt'])
            code = extract_python_code(result['response'])

            record = {
                'prompt_id': prompt_id,
                'model_id': model_id,
                'prompt': prompt['prompt'],
                'code_smells_target': prompt['code_smells'],
                'complexity': prompt['complexity'],
                'domain': prompt['domain'],
                'raw_response': result['response'],
                'extracted_code': code,
                'input_tokens': result['input_tokens'],
                'output_tokens': result['output_tokens'],
                'generation_time_s': result['generation_time_s'],
                'tokens_per_second': result['tokens_per_second'],
                'timestamp': datetime.now().isoformat(),
                'error': None,
            }

            with open(results_file, 'a') as f:
                f.write(json.dumps(record) + '\n')

            with open(model_dir / 'code' / f'{prompt_id}.py', 'w') as f:
                f.write(code)

            print(f"OK ({result['output_tokens']} tok, {result['generation_time_s']}s, {result['tokens_per_second']} tok/s)")

        except Exception as e:
            error_msg = f"{type(e).__name__}: {e}"
            print(f"ERROR: {error_msg}")
            errors.append({'prompt_id': prompt_id, 'error': error_msg})

            record = {
                'prompt_id': prompt_id, 'model_id': model_id,
                'prompt': prompt['prompt'],
                'code_smells_target': prompt['code_smells'],
                'complexity': prompt['complexity'],
                'domain': prompt['domain'],
                'raw_response': None, 'extracted_code': None,
                'input_tokens': 0, 'output_tokens': 0,
                'generation_time_s': 0, 'tokens_per_second': 0,
                'timestamp': datetime.now().isoformat(),
                'error': error_msg,
            }
            with open(results_file, 'a') as f:
                f.write(json.dumps(record) + '\n')

        # Sync to Drive every 50 prompts
        if (i + 1) % 50 == 0:
            sync_to_drive(model_id)
            print(f"  >> Synced to Drive ({len(completed_ids)+i+1} done)")

    # Save run summary
    summary = {
        'model_id': model_id,
        'total_prompts': len(ALL_PROMPTS),
        'completed': len(ALL_PROMPTS) - len(remaining) + len(remaining) - len(errors),
        'errors': len(errors),
        'error_details': errors,
        'run_finished': datetime.now().isoformat(),
    }
    with open(model_dir / 'run_summary.json', 'w') as f:
        json.dump(summary, f, indent=2)

    # Sync final results to Drive
    sync_to_drive(model_id)

    print(f"\n  DONE: {model_id} — {summary['completed']} completed, {summary['errors']} errors")
    print(f"  Saved to Drive: {DRIVE_OUTPUT}/{model_id}")

    # Free GPU for next model
    del model, tokenizer
    free_gpu()

print(f"\n{'='*60}")
print("ALL MODELS COMPLETE")
print(f"{'='*60}")

## 5. Run Code Smell Detection

Analyze all generated code with the heuristic smell detector.

In [ ]:
from detector.smell_detector import detect_all_smells

# Load prompt metadata for accuracy checking
prompt_lookup = {p['id']: p for p in ALL_PROMPTS}

all_model_summaries = []

for model_dir in sorted(Path(LOCAL_OUTPUT).iterdir()):
    if not model_dir.is_dir():
        continue
    code_dir = model_dir / 'code'
    if not code_dir.exists():
        continue

    model_id = model_dir.name
    code_files = sorted(code_dir.glob('*.py'))
    if not code_files:
        continue

    print(f"\nDetecting smells: {model_id} ({len(code_files)} files)")

    detections = []
    stats = {
        'model_id': model_id,
        'total_files': len(code_files),
        'files_with_smells': 0,
        'total_smells': 0,
        'smell_type_counts': {},
        'by_complexity': {'basic': 0, 'intermediate': 0, 'advanced': 0},
        'target_matches': 0,
        'target_total': 0,
    }

    for code_file in code_files:
        prompt_id = code_file.stem
        prompt_info = prompt_lookup.get(prompt_id, {})
        target_smells = prompt_info.get('code_smells', [])
        complexity = prompt_info.get('complexity', 'unknown')

        try:
            code = code_file.read_text(encoding='utf-8')
        except Exception:
            continue
        if not code.strip():
            continue

        result = detect_all_smells(code)

        detections.append({
            'prompt_id': prompt_id,
            'model_id': model_id,
            'target_smells': target_smells,
            'complexity': complexity,
            'domain': prompt_info.get('domain', 'unknown'),
            'smells_detected': result['smell_types_detected'],
            'total_smells': result['total_smells'],
            'smell_details': result['smells'],
            'code_lines': len(code.splitlines()),
        })

        if result['total_smells'] > 0:
            stats['files_with_smells'] += 1
        stats['total_smells'] += result['total_smells']
        stats['by_complexity'][complexity] = stats['by_complexity'].get(complexity, 0) + result['total_smells']

        for s in result['smell_types_detected']:
            stats['smell_type_counts'][s] = stats['smell_type_counts'].get(s, 0) + 1

        # Check target match
        for target in target_smells:
            stats['target_total'] += 1
            target_l = target.lower()
            detected_l = [s.lower() for s in result['smell_types_detected']]
            if any(target_l in d or d in target_l for d in detected_l):
                stats['target_matches'] += 1

    # Save detections
    with open(model_dir / 'detections.jsonl', 'w') as f:
        for d in detections:
            f.write(json.dumps(d) + '\n')

    stats['target_accuracy_pct'] = round(
        stats['target_matches'] / max(stats['target_total'], 1) * 100, 2
    )
    stats['timestamp'] = datetime.now().isoformat()

    with open(model_dir / 'detection_summary.json', 'w') as f:
        json.dump(stats, f, indent=2)

    all_model_summaries.append(stats)
    sync_to_drive(model_id)

    print(f"  Files with smells: {stats['files_with_smells']}/{stats['total_files']}")
    print(f"  Total smells: {stats['total_smells']}")
    print(f"  Target accuracy: {stats['target_accuracy_pct']}%")

print("\nDetection complete for all models.")

## 6. Comparative Report

In [ ]:
import pandas as pd

# Build comparison table
rows = []
for s in all_model_summaries:
    rows.append({
        'Model': s['model_id'],
        'Files': s['total_files'],
        'Files w/ Smells': s['files_with_smells'],
        'Smell Rate %': round(s['files_with_smells'] / max(s['total_files'], 1) * 100, 1),
        'Total Smells': s['total_smells'],
        'Avg Smells/File': round(s['total_smells'] / max(s['total_files'], 1), 2),
        'Target Accuracy %': s['target_accuracy_pct'],
    })

df = pd.DataFrame(rows).sort_values('Total Smells', ascending=False)
df.index = range(1, len(df) + 1)

print("\n" + "="*80)
print("COMPARATIVE RESULTS — Code Smell Detection Across Models")
print("="*80)
print(df.to_string())

# Save
report = {'generated': datetime.now().isoformat(), 'comparison': rows}
with open(Path(LOCAL_OUTPUT) / 'comparative_report.json', 'w') as f:
    json.dump(report, f, indent=2)

df.to_csv(Path(LOCAL_OUTPUT) / 'comparative_report.csv', index=False)
shutil.copy(Path(LOCAL_OUTPUT) / 'comparative_report.json', Path(DRIVE_OUTPUT) / 'comparative_report.json')
shutil.copy(Path(LOCAL_OUTPUT) / 'comparative_report.csv', Path(DRIVE_OUTPUT) / 'comparative_report.csv')

print(f"\nReports saved to Google Drive: {DRIVE_OUTPUT}")

In [ ]:
# Smell distribution chart
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(18, 6))

# Chart 1: Total smells by model
models = [r['Model'] for r in rows]
smells = [r['Total Smells'] for r in rows]
axes[0].barh(models, smells, color='steelblue')
axes[0].set_xlabel('Total Smells Detected')
axes[0].set_title('Code Smells Generated by Model')
axes[0].invert_yaxis()

# Chart 2: Target accuracy by model
accuracy = [r['Target Accuracy %'] for r in rows]
axes[1].barh(models, accuracy, color='coral')
axes[1].set_xlabel('Target Smell Match %')
axes[1].set_title('How Often Models Produced the Requested Smell')
axes[1].invert_yaxis()

plt.tight_layout()
plt.savefig(Path(LOCAL_OUTPUT) / 'comparison_chart.png', dpi=150, bbox_inches='tight')
shutil.copy(Path(LOCAL_OUTPUT) / 'comparison_chart.png', Path(DRIVE_OUTPUT) / 'comparison_chart.png')
plt.show()
print('Chart saved to Drive.')

## 7. Download Results

Results are already on Google Drive at `My Drive/code-smell-outputs/`. You can also download directly:

In [ ]:
# Zip all outputs for download
!cd /content/code-smell && zip -r /content/code-smell-outputs.zip outputs/ -x '*.pyc'

from google.colab import files
files.download('/content/code-smell-outputs.zip')
print('Download started!')